# Survival Stacking avec XGBoost Classifier

Ce notebook développe, entraîne et évalue un modèle **Survival Stacking** basé sur un **classifieur XGBoost** afin de prédire le temps restant avant l'insuffisance rénale terminale (IRT) chez les patients atteints de polykystose rénale autosomique dominante (PKRAD).

À partir des jeux de données dynamiques préparés précédemment, le problème de survie est transformé en une tâche de classification multiclasse à l'aide de l'approche **Survival Stacking**. Les données sont ensuite préparées pour l'entraînement du modèle, les hyperparamètres sont optimisés à l'aide d'Optuna, puis le modèle est entraîné et évalué sur le jeu de validation interne.

Les performances prédictives sont analysées à l'aide de plusieurs métriques de classification, notamment l'accuracy, la balanced accuracy et les scores F1. Une analyse est également réalisée en fonction du nombre de visites disponibles afin d'étudier l'évolution des performances lorsque de nouvelles mesures longitudinales deviennent disponibles.

Enfin, une analyse d'interprétabilité basée sur les valeurs SHAP est réalisée afin d'identifier les variables les plus influentes dans les prédictions et de mieux comprendre le comportement du modèle. Une validation externe est ensuite réalisée sur une cohorte indépendante afin d'évaluer la capacité de généralisation du modèle.


In [ ]:
# ==========================================================
# LIBRAIRIES
# ==========================================================

import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import shap

import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

import optuna

from xgboost import XGBClassifier

from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

from sksurv.ensemble import RandomSurvivalForest
from sksurv.preprocessing import OneHotEncoder
from sksurv.util import Surv

# ==========================================================
# CONFIGURATION
# ==========================================================

pl.Config.set_tbl_rows(20)

## Importation des données

In [ ]:
# ==================================================
# IMPORTATION DU JEU DE DONNEES
# ==================================================

train_dynamic = pl.read_parquet("My_data/train_dynamic.parquet")

internal_test_dynamic = pl.read_parquet("My_data/internal_test_dynamic.parquet")

external_test_dynamic = pl.read_parquet("My_data/external_test_dynamic.parquet")

train_dynamic.head()

In [ ]:
# =============================================================================
# Définition des intervalles de temps (années)
# =============================================================================

INTERVALS = [
    (0, "0-5",    0.0, 5.0),
    (1, "5-10",   5.0, 10.0),
    (2, "10-15", 10.0, 15.0),
    (3, ">15",   15.0, float("inf")),
]


# =============================================================================
# Construction du dataset Snapshot-Period
# =============================================================================

def build_snapshot_period_dataset(df: pl.DataFrame) -> pl.DataFrame:
    """
    Convertit un dataset longitudinal (une ligne = un snapshot)
    en dataset Snapshot-Period pour une analyse de survie discrète.

    Chaque snapshot est dupliqué jusqu'à l'intervalle contenant
    l'événement ou la censure.

    Les intervalles complètement traversés reçoivent :
        event_interval = 0

    L'intervalle terminal reçoit :
        event_interval = event_irt

    Les intervalles suivants ne sont pas créés.
    """

    stacked_rows = []

    for row in df.iter_rows(named=True):

        time = row["time_to_event"]
        event = row["event_irt"]

        if time < 0:
            raise ValueError(
                f"time_to_event négatif pour le patient {row['patient_id']}"
            )

        # --------------------------------------------------------------
        # Recherche de l'intervalle terminal
        # --------------------------------------------------------------

        target_id = None

        for interval_id, _, start, end in INTERVALS:

            if start <= time < end:
                target_id = interval_id
                break

        if target_id is None:
            raise ValueError(
                f"Aucun intervalle trouvé pour time_to_event={time}"
            )

        # --------------------------------------------------------------
        # Construction du dataset Snapshot-Period
        # --------------------------------------------------------------

        for interval_id, label, start, end in INTERVALS:

            if interval_id > target_id:
                break

            new_row = row.copy()

            new_row["interval_id"] = interval_id
            new_row["interval_label"] = label
            new_row["interval_start"] = start
            new_row["interval_end"] = end

            new_row["is_last_interval"] = (
                interval_id == target_id
            )

            new_row["event_interval"] = (
                event if interval_id == target_id else 0
            )

            stacked_rows.append(new_row)

    return (
        pl.DataFrame(stacked_rows)
        .sort(
            ["patient_id", "creat_date", "interval_id"]
        )
    )

In [ ]:
# ================================================================
# CONVERSION DES SNAPSHOTS EN DONNÉES SURVIVAL STACKING
# ================================================================

# Transformation du jeu d'entraînement
# (utilisé pour Optuna, la validation croisée et l'entraînement final)
train_snapshot_period = build_snapshot_period_dataset(train_dynamic)

# Transformation du jeu de test interne
# (utilisé uniquement pour les prédictions et l'évaluation du moèle)
internal_test_snapshot_period = build_snapshot_period_dataset(
    internal_test_dynamic
)

# Transformation du jeu de test interne
# (utilisé uniquement pour l'évaluation finale)
external_test_snapshot_period = build_snapshot_period_dataset(
    external_test_dynamic
)

In [ ]:
# ================================================================
# VARIABLES DISPONIBLES AU MOMENT DU SNAPSHOT
# ================================================================

FEATURES = [

    # --------------------------
    # Démographie
    # --------------------------
    "sex",
    "genetic",

    # --------------------------
    # Snapshot actuel
    # --------------------------
    "age_creat",
    "current_ckd_epi",

    # --------------------------
    # Etat clinique actuel
    # --------------------------
    "hypertension_at_t",
    "age_hypertension_at_t",

    "urologic_event_at_t",
    "age_urologic_event_at_t",

    # --------------------------
    # Trajectoire eGFR
    # --------------------------
    "baseline_ckd_epi",
    "previous_ckd_epi",
    "delta_ckd_from_baseline",

    "recent_annualized_change",

    "egfr_mean",
    "egfr_std",
    "egfr_slope",
    "recent_delta_ckd",
    "recent_egfr_slope",

    # --------------------------
    # Intensité du suivi
    # --------------------------
    "time_since_last_measure",
    "followup_density",
    "n_measures",

    # --------------------------
    # Intervalle de survie
    # --------------------------
    "interval_id",
]

In [ ]:
# ================================================================
# PRÉPARATION DES VARIABLES ET ENCODAGE DES CATÉGORIES
# ================================================================

def prepare_snapshot_period_data(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    features: list[str],
):
    """
    Prépare les données pour le Survival Stacking avec XGBoost.

    Cette fonction :

    - sélectionne les variables d'intérêt ;
    - encode les variables catégorielles (sex, genetic)
      uniquement à partir du jeu d'entraînement ;
    - applique les mêmes encodeurs au jeu de test ;
    - conserve toutes les autres variables inchangées.

    Parameters
    ----------
    train_df : pl.DataFrame
        Jeu d'entraînement Snapshot-Period.

    test_df : pl.DataFrame
        Jeu de test Snapshot-Period.

    features : list[str]
        Variables utilisées par le modèle.

    Returns
    -------
    X_train : pandas.DataFrame

    y_train : pandas.Series

    X_test : pandas.DataFrame

    y_test : pandas.Series

    encoders : dict
        Encodeurs utilisés pour les variables catégorielles.
    """

    # ------------------------------------------------------------------
    # Conversion Polars -> pandas
    # ------------------------------------------------------------------

    train = train_df.to_pandas().copy()
    test = test_df.to_pandas().copy()

    # ------------------------------------------------------------------
    # Sélection des variables utilisées par le modèle
    # ------------------------------------------------------------------

    X_train = train[features].copy()
    X_test = test[features].copy()

    # ------------------------------------------------------------------
    # Extraction de la cible (événement dans l'intervalle)
    # ------------------------------------------------------------------

    y_train = train["event_interval"].astype(int)
    y_test = test["event_interval"].astype(int)

    # ------------------------------------------------------------------
    # Encodage des variables catégorielles appris sur le train
    # ------------------------------------------------------------------

    encoders = {}

    categorical_features = [
        "sex",
        "genetic",
    ]

    for col in categorical_features:

        encoder = LabelEncoder()

        # L'encodeur est appris uniquement sur le train
        encoder.fit(
            X_train[col].astype(str)
        )

        # Vérification des modalités du test
        unknown = (
            set(X_test[col].astype(str))
            - set(encoder.classes_)
        )

        if unknown:
            raise ValueError(
                f"Nouvelles modalités détectées pour '{col}' : {unknown}"
            )

        X_train[col] = encoder.transform(
            X_train[col].astype(str)
        )

        # Le test est transformé avec le même encodeur

        X_test[col] = encoder.transform(
            X_test[col].astype(str)
        )

        encoders[col] = encoder

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        encoders,
    )

In [ ]:
# ==========================================================
# PARAMÈTRES PAR DÉFAUT DU XGBCLASSIFIER
# ===========================================================

DEFAULT_XGB_SURVIVAL_STACKING_PARAMS = {

    # ------------------------------------------------------
    # Boosting
    # ------------------------------------------------------
    "learning_rate": 0.05,
    "max_depth": 4,
    "min_child_weight": 1.0,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "gamma": 0.0,

    # ------------------------------------------------------
    # Régularisation
    # ------------------------------------------------------
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    # ------------------------------------------------------
    # Déséquilibre des classes
    # ------------------------------------------------------
    "scale_pos_weight": 1.0,

    # ------------------------------------------------------
    # Nombre maximal d'arbres
    # (early stopping arrêtera avant)
    # ------------------------------------------------------
    "n_estimators": 2000,
}


# ==========================================================
# CONSTRUCTION DU MODÈLE XGB SURVIVAL STACKING
# ==========================================================

def build_xgb_survival_stacking_model(
    random_state: int = 42,
    n_jobs: int = -1,
    **params,
) -> XGBClassifier:
    """
    Construit un modèle XGBoost pour le Survival Stacking.

    Le modèle estime le hazard conditionnel
    P(T ∈ intervalle | T >= début de l'intervalle).

    Notes
    -----
    Cette fonction construit uniquement le modèle.

    L'entraînement (fit), l'early stopping et l'évaluation
    sont réalisés dans train_xgb_survival_stacking().
    """

    # ======================================================
    # Fusion des paramètres
    # ======================================================

    model_params = DEFAULT_XGB_SURVIVAL_STACKING_PARAMS.copy()
    model_params.update(params)

    # ======================================================
    # Construction du modèle
    # ======================================================

    model = XGBClassifier(

        # --------------------------------------------------
        # Objectif
        # --------------------------------------------------
        objective="binary:logistic",

        # --------------------------------------------------
        # Algorithme de boosting
        # --------------------------------------------------
        booster="gbtree",

        # --------------------------------------------------
        # Hyperparamètres
        # --------------------------------------------------
        **model_params,

        # --------------------------------------------------
        # Métrique d'entraînement
        # --------------------------------------------------
        eval_metric="logloss",

        # --------------------------------------------------
        # Reproductibilité
        # --------------------------------------------------
        random_state=random_state,

        # --------------------------------------------------
        # Calcul
        # --------------------------------------------------
        tree_method="hist",
        n_jobs=n_jobs,

        # --------------------------------------------------
        # Affichage
        # --------------------------------------------------
        verbosity=0,
    )

    return model

In [ ]:
# ==========================================================
# ENTRAÎNEMENT DU MODÈLE XGB SURVIVAL STACKING
# ==========================================================

def fit_xgb_survival_stacking_model(
    model: XGBClassifier,
    X_train,
    y_train,
    X_valid=None,
    y_valid=None,
):
    """
    Entraîne un modèle XGBoost pour le Survival Stacking.

    Parameters
    ----------
    model : XGBClassifier
        Modèle XGBoost.

    X_train : pandas.DataFrame
        Variables d'entraînement.

    y_train : pandas.Series
        Variable cible d'entraînement.

    X_valid : pandas.DataFrame, optional
        Variables de validation.

    y_valid : pandas.Series, optional
        Variable cible de validation.

    Returns
    -------
    model : XGBClassifier
        Modèle entraîné.

    history : dict
        Historique d'entraînement. Vide si aucun jeu de validation
        n'est fourni.

    best_iteration : int | None
        Meilleure itération sélectionnée par XGBoost.
        None si aucun jeu de validation n'est utilisé.

    best_score : float | None
        Meilleur score de validation.
        None si aucun jeu de validation n'est utilisé.
    """

    # ==========================================================
    # Entraînement du modèle final sur l'ensemble du jeu d'entraînement
    # ==========================================================

    if X_valid is None or y_valid is None:

        model.fit(
            X_train,
            y_train,
            verbose=False,
        )

        history = {}
        best_iteration = None
        best_score = None

    # ==========================================================
    # Entraînement avec validation (Optuna / CV)
    # ==========================================================

    else:

        model.fit(
            X_train,
            y_train,
            eval_set=[
                (X_train, y_train),
                (X_valid, y_valid),
            ],
            verbose=False,
        )

        history = model.evals_result()

        best_iteration = getattr(
            model,
            "best_iteration",
            None,
        )

        best_score = getattr(
            model,
            "best_score",
            None,
        )

    return (
        model,
        history,
        best_iteration,
        best_score,
    )

In [ ]:
# ==========================================================
# PRÉDICTION DU HAZARD CONDITIONNEL PAR INTERVALLE
# ==========================================================

def predict_conditional_hazard(
    model: XGBClassifier,
    X: pd.DataFrame,
    data_snapshot_period: pl.DataFrame,
):
    """
    Prédit les hazards conditionnels du Survival Stacking.
    """

    # ======================================================
    # Vérifications
    # ======================================================

    required_columns = [

        "patient_id",

        "n_measures",

        "interval_id",

        "interval_label",

        "interval_start",

        "interval_end",

    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in data_snapshot_period.columns
    ]

    if missing_columns:

        raise ValueError(
            "Colonnes manquantes : "
            + ", ".join(missing_columns)
        )

    # ======================================================================
    # Prédiction du hazard conditionnel pour chaque ligne Snapshot-Period
    # ======================================================================

    conditional_hazard = model.predict_proba(X)[:, 1]

    # =======================================================================
    # Association des probabilités aux intervalles correspondants
    # =======================================================================

    hazards = (
        data_snapshot_period
        .select(required_columns)
        .with_columns(
            pl.Series(
                "conditional_hazard",
                conditional_hazard,
            )
        )
        .sort(
            [
                "patient_id",
                "n_measures",
                "interval_id",
            ]
        )
    )

    return hazards

In [ ]:
# ===================================================================
# RECONSTRUCTION DE LA SURVIE ET DES PROBABILITÉS PAR INTERVALLE
# ===================================================================

def compute_survival_probabilities(
    hazards: pl.DataFrame,
):
    """
    Reconstruit la fonction de survie à partir des hazards
    conditionnels prédits par le modèle Survival Stacking.
    """

    # ==========================================================
    # Vérification des colonnes
    # ==========================================================

    required_columns = [

        "patient_id",
        "n_measures",
        "interval_id",
        "interval_label",
        "interval_start",
        "interval_end",
        "conditional_hazard",

    ]

    missing_columns = [

        col
        for col in required_columns
        if col not in hazards.columns

    ]

    if missing_columns:

        raise ValueError(
            "Colonnes manquantes : "
            + ", ".join(missing_columns)
        )

    # ==========================================================
    # Tri
    # ==========================================================

    survival = hazards.sort(
        [
            "patient_id",
            "n_measures",
            "interval_id",
        ]
    )

    # ==========================================================
    # Vérification des doublons
    # ==========================================================

    if survival.select(
        pl.struct(
            [
                "patient_id",
                "n_measures",
                "interval_id",
            ]
        ).is_duplicated().any()
    ).item():

        raise ValueError(
            "Des doublons patient/snapshot/interval_id ont été détectés."
        )

    # ==========================================================
    # Sécurité numérique
    # ==========================================================

    EPS = 1e-12

    survival = survival.with_columns(

        pl.col("conditional_hazard")
        .clip(EPS, 1.0 - EPS)

    )

    # ==========================================================
    # Complément du hazard
    # ==========================================================

    survival = survival.with_columns(

        (
            1.0 - pl.col("conditional_hazard")
        ).alias("one_minus_hazard")

    )

    # ==========================================================
    # Reconstruction de la fonction de survie S(t)
    # ==========================================================

    survival = survival.with_columns(

        pl.col("one_minus_hazard")
        .cum_prod()
        .over(
            [
                "patient_id",
                "n_measures",
            ]
        )
        .alias("survival_probability")

    )

    # ==========================================================
    # Survie avant l'intervalle
    # ==========================================================

    survival = survival.with_columns(

        pl.col("survival_probability")
        .shift(1)
        .fill_null(1.0)
        .over(
            [
                "patient_id",
                "n_measures",
            ]
        )
        .alias("previous_survival")

    )

    # ==========================================================
    # Reconstruction de P(T ∈ intervalle)
    # ==========================================================

    survival = survival.with_columns(

        (
            pl.col("previous_survival")
            * pl.col("conditional_hazard")
        ).alias("interval_probability")

    )

    # ==========================================================
    # Vérification de cohérence
    # ==========================================================

    probability_check = (

        survival

        .group_by(
            [
                "patient_id",
                "n_measures",
            ]
        )

        .agg(

            pl.sum("interval_probability").alias(
                "interval_probability_sum"
            ),

            pl.last("survival_probability").alias(
                "final_survival"
            ),

        )

        .with_columns(

            (
                pl.col("interval_probability_sum")
                + pl.col("final_survival")
            ).alias("total_probability")

        )

    )

    if not np.allclose(

        probability_check["total_probability"].to_numpy(),

        1.0,

        atol=1e-6,

    ):

        raise ValueError(
            "Les probabilités reconstruites ne somment pas à 1."
        )

    # ==========================================================
    # Nettoyage
    # ==========================================================

    survival = survival.drop(
        "one_minus_hazard",
        "previous_survival",
    )

    return survival

In [ ]:
# ==========================================================
# CONSTRUCTION DE LA MATRICE DES PROBABILITÉS PAR INTERVALLE
# ==========================================================

def build_interval_probability_matrix(
    survival: pd.DataFrame,
):
    """
    Construit une matrice contenant, pour chaque patient et chaque snapshot,
    les probabilités prédites de chaque intervalle ainsi que l'intervalle
    le plus probable et sa probabilité associée.

    Parameters
    ----------
    survival : pandas.DataFrame

        Sortie de compute_survival_probabilities().

    Returns
    -------
    probability_matrix : pandas.DataFrame

        Une ligne par patient et snapshot contenant :

            patient_id
            snapshot
            P(0-5)
            P(5-10)
            P(10-15)
            P(>15)
            predicted_interval
            predicted_probability
    """

    # ==========================================================
    # Vérification
    # ==========================================================

    required_columns = [

        "patient_id",
        "n_measures",
        "interval_label",
        "interval_probability",

    ]

    missing_columns = [

        c
        for c in required_columns
        if c not in survival.columns

    ]

    if missing_columns:

        raise ValueError(

            "Colonnes manquantes : "

            + ", ".join(missing_columns)

        )

    # ==========================================================
    # Transformation en une ligne par patient et snapshot
    # ==========================================================

    probability_matrix = (

        survival

        .pivot(

            index=[
                "patient_id",
                "n_measures",
            ],

            columns="interval_label",

            values="interval_probability",

        )

        .reset_index()

    )

    # ==========================================================
    # Nom des colonnes
    # ==========================================================

    probability_matrix.columns.name = None

    interval_columns = [

        c
        for c in probability_matrix.columns
        if c not in [

            "patient_id",

            "n_measures",

        ]

    ]

    probability_matrix = probability_matrix.rename(

        columns={

            c: f"P({c})"

            for c in interval_columns

        }

    )

    # ==========================================================
    # Valeurs manquantes
    # ==========================================================

    probability_matrix = probability_matrix.fillna(0.0)

    # ==========================================================
    # Intervalle présentant la probabilité maximale
    # ==========================================================

    probability_columns = [

        c

        for c in probability_matrix.columns

        if c.startswith("P(")

    ]

    probability_matrix["predicted_interval"] = (

        probability_matrix[

            probability_columns

        ]

        .idxmax(axis=1)

        .str.replace(

            "P(",

            "",

            regex=False,

        )

        .str.replace(

            ")",

            "",

            regex=False,

        )

    )

    # ==========================================================
    # Probabilité maximale
    # ==========================================================

    probability_matrix["predicted_probability"] = (

        probability_matrix[

            probability_columns

        ]

        .max(axis=1)

    )

    return probability_matrix

In [ ]:
# ==========================================================
# SÉLECTION DE L'INTERVALLE LE PLUS PROBABLE
# ==========================================================

def predict_survival_interval(
    survival: pl.DataFrame,
):
    """
    Détermine l'intervalle de temps prédit pour chaque
    patient et chaque snapshot.
    """

    # ==========================================================
    # Vérification
    # ==========================================================

    required_columns = [

        "patient_id",
        "n_measures",
        "interval_id",
        "interval_label",
        "interval_probability",

    ]

    missing_columns = [

        col
        for col in required_columns
        if col not in survival.columns

    ]

    if missing_columns:

        raise ValueError(
            "Colonnes manquantes : "
            + ", ".join(missing_columns)
        )

    # =================================================================
    # Sélection de l'intervalle présentant la probabilité maximale
    # =================================================================

    predictions = (

        survival

        .sort(
            [
                "patient_id",
                "n_measures",
                "interval_probability",
            ],
            descending=[False, False, True],
        )

        .group_by(
            [
                "patient_id",
                "n_measures",
            ]
        )

        .first()

        .select(

            [

                "patient_id",

                "n_measures",

                pl.col("interval_id").alias(
                    "predicted_interval_id"
                ),

                pl.col("interval_label").alias(
                    "predicted_interval_label"
                ),

                pl.col("interval_probability").alias(
                    "predicted_probability"
                ),

            ]

        )

        .sort(
            [
                "patient_id",
                "n_measures",
            ]
        )

    )

    return predictions

In [ ]:
# ==========================================================
# ÉVALUATION DES PRÉDICTIONS FINALES
# ==========================================================

def evaluate_survival_stacking_predictions(
    truth: pl.DataFrame,
    predictions: pl.DataFrame,
):
    """
    Évalue les prédictions du modèle Survival Stacking.

    Parameters
    ----------
    truth : pl.DataFrame

        Une ligne par patient et snapshot contenant :

            patient_id
            n_measures
            interval_id
            interval_label

    predictions : pl.DataFrame

        Sortie de predict_survival_interval().

            patient_id
            n_measures
            predicted_interval_id
            predicted_interval_label
            predicted_probability

    Returns
    -------
    metrics : dict

        accuracy
        balanced_accuracy
        macro_f1
        weighted_f1
        confusion_matrix
        classification_report

    results : pl.DataFrame

        Fusion vérité terrain / prédictions.
    """

    # ==========================================================
    # Vérifications
    # ==========================================================

    required_truth = [

        "patient_id",
        "n_measures",
        "interval_id",
        "interval_label",

    ]

    required_predictions = [

        "patient_id",
        "n_measures",
        "predicted_interval_id",
        "predicted_interval_label",
        "predicted_probability",

    ]

    missing_truth = [

        c
        for c in required_truth
        if c not in truth.columns

    ]

    if missing_truth:

        raise ValueError(
            "Colonnes manquantes dans truth : "
            + ", ".join(missing_truth)
        )

    missing_predictions = [

        c
        for c in required_predictions
        if c not in predictions.columns

    ]

    if missing_predictions:

        raise ValueError(
            "Colonnes manquantes dans predictions : "
            + ", ".join(missing_predictions)
        )

    # ==========================================================
    # Fusion
    # ==========================================================

    results = (

        truth

        .join(

            predictions,

            on=[
                "patient_id",
                "n_measures",
            ],

            how="inner",

        )

        .sort(
            [
                "patient_id",
                "n_measures",
            ]
        )

    )

    # ==========================================================
    # Vérification
    # ==========================================================

    if results.height != truth.height:

        raise ValueError(
            "Certaines prédictions sont manquantes."
        )

    # ==========================================================
    # Vérité / prédiction
    # ==========================================================

    y_true = results["interval_id"].to_numpy()

    y_pred = results["predicted_interval_id"].to_numpy()

    # ==========================================================
    # Métriques
    # ==========================================================

    metrics = {

        "accuracy":

            accuracy_score(
                y_true,
                y_pred,
            ),

        "balanced_accuracy":

            balanced_accuracy_score(
                y_true,
                y_pred,
            ),

        "macro_f1":

            f1_score(
                y_true,
                y_pred,
                average="macro",
            ),

        "weighted_f1":

            f1_score(
                y_true,
                y_pred,
                average="weighted",
            ),

        "confusion_matrix":

            confusion_matrix(
                y_true,
                y_pred,
            ),

        "classification_report":

            classification_report(
                y_true,
                y_pred,
                output_dict=True,
                zero_division=0,
            ),

    }

    return (
        metrics,
        results,
    )

In [ ]:
# ==========================================================
# ÉVALUATION DES PATIENTS CENSURÉS
# ==========================================================

def evaluate_censoring_predictions(
    truth: pl.DataFrame,
    predictions: pl.DataFrame,
):
    """
    Évalue la cohérence des prédictions pour les patients censurés.

    Une prédiction est considérée comme une violation si l'intervalle
    prédit se termine strictement avant le temps de censure.

    Parameters
    ----------
    truth : pl.DataFrame

        Dataset Snapshot-Period contenant :

            patient_id
            n_measures
            event_irt
            time_to_event
            is_last_interval

    predictions : pl.DataFrame

        Sortie de predict_survival_interval().

            patient_id
            n_measures
            predicted_interval_id
            predicted_interval_label
            predicted_probability

    Returns
    -------
    metrics : dict

        n_censored
        n_violations
        censoring_violation_rate
        censoring_consistency

    results : pl.DataFrame

        Fusion vérité terrain / prédictions avec
        l'indicateur de violation.
    """

    # ==========================================================
    # Vérifications
    # ==========================================================

    required_truth = [

        "patient_id",
        "n_measures",
        "event_irt",
        "time_to_event",
        "is_last_interval",

    ]

    required_predictions = [

        "patient_id",
        "n_measures",
        "predicted_interval_id",
        "predicted_interval_label",
        "predicted_probability",

    ]

    missing_truth = [

        c
        for c in required_truth
        if c not in truth.columns

    ]

    if missing_truth:

        raise ValueError(
            "Colonnes manquantes dans truth : "
            + ", ".join(missing_truth)
        )

    missing_predictions = [

        c
        for c in required_predictions
        if c not in predictions.columns

    ]

    if missing_predictions:

        raise ValueError(
            "Colonnes manquantes dans predictions : "
            + ", ".join(missing_predictions)
        )

    # ==========================================================
    # Patients censurés (une ligne par patient et snapshot)
    # ==========================================================

    truth = (

        truth

        .filter(

            (pl.col("event_irt") == 0)

            &

            (pl.col("is_last_interval"))

        )

    )

    # ==========================================================
    # Fusion
    # ==========================================================

    results = (

        truth

        .join(

            predictions,

            on=[
                "patient_id",
                "n_measures",
            ],

            how="inner",

        )

    )

    # ==========================================================
    # Fin des intervalles prédits
    # ==========================================================

    interval_mapping = pl.DataFrame({

        "predicted_interval_id": [
            interval_id
            for interval_id, _, _, _ in INTERVALS
        ],

        "predicted_interval_end": [
            float(interval_end)
            for _, _, _, interval_end in INTERVALS
        ],

    })

    results = results.join(

        interval_mapping,

        on="predicted_interval_id",

        how="left",

    )

    # ==========================================================
    # Violation de la censure
    # ==========================================================

    results = results.with_columns(

        (
            pl.col("predicted_interval_end")
            < pl.col("time_to_event")
        ).alias("violation")

    )

    # ==========================================================
    # Métriques
    # ==========================================================

    n_censored = results.height

    n_violations = results.filter(
        pl.col("violation")
    ).height

    violation_rate = (

        n_violations / n_censored

        if n_censored > 0

        else np.nan

    )

    metrics = {

        "n_censored": n_censored,

        "n_violations": n_violations,

        "censoring_violation_rate": violation_rate,

        "censoring_consistency": (

            1.0 - violation_rate

            if n_censored > 0

            else np.nan

        ),

    }

    return (
        metrics,
        results,
    )

In [ ]:
# ==========================================================
# VALIDATION CROISÉE ET OPTIMISATION DES HYPERPARAMÈTRES
# ==========================================================
def objective(
    trial,
    snapshot_period_data,
    features,
    n_splits=5,
):
    """
    Fonction objectif utilisée par Optuna pour optimiser
    le modèle XGBoost Survival Stacking.
    """

    # ==========================================================
    # Conversion éventuelle en pandas
    # ==========================================================

    if isinstance(snapshot_period_data, pl.DataFrame):
        snapshot_period_data = snapshot_period_data.to_pandas()

    # ==========================================================
    # Hyperparamètres
    # ==========================================================

    params = {

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
            log=True,
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8,
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            20,
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0,
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0.0,
            5.0,
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10.0,
            log=True,
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1000,
        ),

    }

    # ==========================================================
    # Validation croisée
    # ==========================================================

    X = snapshot_period_data[features]

    y = snapshot_period_data["interval_id"]

    groups = snapshot_period_data["patient_id"]

    cv = StratifiedGroupKFold(

        n_splits=n_splits,

        shuffle=True,

        random_state=42,

    )

    scores = []

    # ==========================================================
    # Boucle de validation croisée
    # ==========================================================

    for train_idx, valid_idx in cv.split(
        X,
        y,
        groups=groups,
    ):

        train_df = (
            snapshot_period_data
            .iloc[train_idx]
            .reset_index(drop=True)
        )

        valid_df = (
            snapshot_period_data
            .iloc[valid_idx]
            .reset_index(drop=True)
        )

        # ------------------------------------------------------
        # Conversion vers Polars
        # ------------------------------------------------------

        train_df_pl = pl.from_pandas(train_df)

        valid_df_pl = pl.from_pandas(valid_df)

        # ------------------------------------------------------
        # Préparation des données
        # ------------------------------------------------------

        (
            X_train,
            y_train,
            X_valid,
            y_valid,
            _,
        ) = prepare_snapshot_period_data(

            train_df=train_df_pl,

            test_df=valid_df_pl,

            features=features,

        )

        # ------------------------------------------------------
        # Construction du modèle
        # ------------------------------------------------------

        model = build_xgb_survival_stacking_model(
            **params,
        )

        # ------------------------------------------------------
        # Entraînement
        # ------------------------------------------------------

        model, _, _, _ = fit_xgb_survival_stacking_model(

            model=model,

            X_train=X_train,
            y_train=y_train,

            X_valid=X_valid,
            y_valid=y_valid,

        )

        # ------------------------------------------------------
        # Hazards conditionnels
        # ------------------------------------------------------

        hazards = predict_conditional_hazard(

            model=model,

            X=X_valid,

            data_snapshot_period=valid_df_pl,

        )

        # ------------------------------------------------------
        # Probabilités de survie
        # ------------------------------------------------------

        survival = compute_survival_probabilities(
            hazards,
        )

        # ------------------------------------------------------
        # Intervalle prédit
        # ------------------------------------------------------

        predictions = predict_survival_interval(
            survival,
        )

        # ------------------------------------------------------
        # Vérité terrain
        # ------------------------------------------------------

        truth = (

            valid_df_pl

            .filter(
                pl.col("event_interval") == 1
            )

            .select(
                [
                    "patient_id",
                    "n_measures",
                    "interval_id",
                    "interval_label",
                ]
            )

            .unique(
                subset=[
                    "patient_id",
                    "n_measures",
                ]
            )

            .sort(
                [
                    "patient_id",
                    "n_measures",
                ]
            )

        )

        # ------------------------------------------------------
        # Évaluation
        # ------------------------------------------------------

        metrics, _ = evaluate_survival_stacking_predictions(

            truth=truth,

            predictions=predictions,

        )

        scores.append(
            metrics["macro_f1"]
        )
        trial.report(
            np.mean(scores),
            step=len(scores),
        )
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))

In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES AVEC OPTUNA
# ==========================================================

def optimize_xgb_survival_stacking_hyperparameters(
    snapshot_period_data,
    features,
    n_trials=50,
    timeout=None,
    n_splits=5,
    study_name="xgb_survival_stacking",
    direction="maximize",
):
    """
    Optimisation des hyperparamètres du modèle
    XGBoost Survival Stacking avec Optuna.

    Parameters
    ----------
    snapshot_period_data : pandas.DataFrame

        Jeu de données au format snapshot-period.

    features : list[str]

        Variables explicatives.

    n_trials : int, default=50

    timeout : int | None

        Temps maximal (secondes).

    n_splits : int, default=5

        Nombre de folds de validation croisée.

    study_name : str

    direction : {"maximize","minimize"}

    Returns
    -------
    best_params : dict

        Meilleurs hyperparamètres.

    study : optuna.study.Study

        Etude Optuna complète.
    """

    # ==========================================================
    # Création de l'étude
    # ==========================================================

    study = optuna.create_study(

        direction=direction,

        study_name=study_name,

        sampler=optuna.samplers.TPESampler(
            seed=42,
        ),

        pruner=optuna.pruners.MedianPruner(),

    )

    # ==========================================================
    # Optimisation
    # ==========================================================

    study.optimize(

        lambda trial: objective(

            trial=trial,

            snapshot_period_data=snapshot_period_data,

            features=features,

            n_splits=n_splits,

        ),

        n_trials=n_trials,

        timeout=timeout,

        show_progress_bar=True,

    )

    # ==========================================================
    # Résultats
    # ==========================================================

    print("=" * 70)
    print("OPTIMISATION TERMINÉE")
    print("=" * 70)

    print(f"Nombre d'essais : {len(study.trials)}")
    print(f"Meilleur Macro-F1 : {study.best_value:.5f}")

    print("\nMeilleurs hyperparamètres :")

    for key, value in study.best_params.items():
        print(f"{key:<25}: {value}")

    print("=" * 70)

    return study.best_params, study

In [ ]:
# ==========================================================
# OPTIMISATION DES HYPERPARAMÈTRES AVEC OPTUNA
# ==========================================================

BEST_XGB_SURVIVAL_STACKING_PARAMS, study = (
    optimize_xgb_survival_stacking_hyperparameters(
        snapshot_period_data=train_snapshot_period,
        features=FEATURES,
        n_trials=50,
    )
)

In [ ]:
# ================================================================
# PRÉPARATION DES MATRICES D'ENTRAÎNEMENT ET DE TEST
# ================================================================

(
    X_train,
    y_train,
    X_test,
    y_test,
    encoders,
) = prepare_snapshot_period_data(
    train_snapshot_period,
    internal_test_snapshot_period,
    FEATURES,
)

In [ ]:
# ==========================================================
# CONSTRUCTION ET ENTRAÎNEMENT DU MODÈLE
# ==========================================================

model = build_xgb_survival_stacking_model(
    **BEST_XGB_SURVIVAL_STACKING_PARAMS,
)

model, history, best_iteration, best_score = (
    fit_xgb_survival_stacking_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
    )
)

In [ ]:
# ==========================================================
# PRÉDICTION SUR LE JEU DE TEST
# ==========================================================

hazards = predict_conditional_hazard(
    model=model,
    X=X_test,
    data_snapshot_period=internal_test_snapshot_period,
)

survival = compute_survival_probabilities(hazards)

predictions = predict_survival_interval(survival)

truth = (
    internal_test_snapshot_period
    .filter(
        pl.col("event_interval") == 1
    )
    .select(
        [
            "patient_id",
            "n_measures",
            "interval_id",
            "interval_label",
        ]
    )
    .unique(
        subset=[
            "patient_id",
            "n_measures",
        ]
    )
    .sort(
        [
            "patient_id",
            "n_measures",
        ]
    )
)


In [ ]:
# ==========================================================
# ÉVALUATION SUR LE JEU DE TEST
# ==========================================================

metrics, results = evaluate_survival_stacking_predictions(
    truth=truth,
    predictions=predictions,
)

# ==========================================================
# PRÉSENTATION DES RÉSULTATS
# ==========================================================

print("=" * 70)
print("SURVIVAL STACKING - RÉSULTATS")
print("=" * 70)

print(f"Accuracy              : {metrics['accuracy']:.4f}")
print(f"Balanced Accuracy     : {metrics['balanced_accuracy']:.4f}")
print(f"Macro F1-score        : {metrics['macro_f1']:.4f}")
print(f"Weighted F1-score     : {metrics['weighted_f1']:.4f}")

print("\n")

# ==========================================================
# Matrice de confusion
# ==========================================================

print("=" * 70)
print("MATRICE DE CONFUSION")
print("=" * 70)

print(metrics["confusion_matrix"])

print("\n")

# ==========================================================
# Rapport de classification
# ==========================================================

print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report = (
    pd.DataFrame(metrics["classification_report"])
    .T
    .round(3)
)

print(report)

print("\n")

# ==========================================================
# Résultats détaillés
# ==========================================================

print("=" * 70)
print("APERÇU DES PRÉDICTIONS")
print("=" * 70)

print(results)

print("\n")

# ==========================================================
# Statistiques
# ==========================================================

print("=" * 70)
print("STATISTIQUES")
print("=" * 70)

print(f"Nombre de prédictions : {results.height}")

print(
    f"Nombre de patients    : "
    f"{results.select('patient_id').n_unique()}"
)

In [ ]:
results.with_columns(
    (
        pl.col("predicted_interval_id")
        == pl.col("interval_id")
    ).alias("correct_prediction")
)

In [ ]:
# ==========================================================
# PATIENTS CENSURÉS
# ==========================================================

censor_metrics, censor_results = evaluate_censoring_predictions(
    truth=internal_test_snapshot_period,
    predictions=predictions,
)

print("=" * 70)
print("ANALYSE DES PATIENTS CENSURÉS")
print("=" * 70)

print(f"Nombre de patients censurés      : {censor_metrics['n_censored']}")
print(f"Violations de la censure         : {censor_metrics['n_violations']}")
print(f"Taux de violation                : {100*censor_metrics['censoring_violation_rate']:.2f}%")
print(f"Cohérence avec la censure        : {100*censor_metrics['censoring_consistency']:.2f}%")

print("\n")

# ==========================================================
# Tableau récapitulatif
# ==========================================================

summary = (

    censor_results

    .select(

        [

            "patient_id",

            "n_measures",

            "time_to_event",

            "predicted_interval_label",

            "predicted_probability",

            "violation",

        ]

    )

    .sort(

        [

            "violation",

            "time_to_event",

        ],

        descending=[True, False],

    )

)

print("=" * 70)
print("PATIENTS CENSURÉS")
print("=" * 70)

print(summary)

print("\n")

# ==========================================================
# Violations uniquement
# ==========================================================

violations = summary.filter(
    pl.col("violation")
)

print("=" * 70)
print("PATIENTS PRÉSENTANT UNE VIOLATION")
print("=" * 70)

if violations.height == 0:

    print("Aucune violation.")

else:

    print(violations)

In [ ]:
# ==========================================================
# ÉVALUATION DES PERFORMANCES PAR SNAPSHOT (1 À 12)
# ==========================================================

def evaluate_survival_stacking_by_snapshot(
    truth: pl.DataFrame,
    predictions: pl.DataFrame,
    snapshot_period: pl.DataFrame,
):
    """
    Évalue les performances du Survival Stacking
    pour les snapshots 1 à 12.

    Parameters
    ----------
    truth : pl.DataFrame
        Vérités terrain (uniquement les événements).

    predictions : pl.DataFrame
        Prédictions du modèle.

    snapshot_period : pl.DataFrame
        Dataset Snapshot-Period complet
        (événements + censures).

    Returns
    -------
    summary : pl.DataFrame
    """

    # ==========================================================
    # Fusion vérité / prédictions
    # ==========================================================

    results = (
        truth
        .join(
            predictions,
            on=[
                "patient_id",
                "n_measures",
            ],
            how="inner",
        )
        .sort(
            [
                "n_measures",
                "patient_id",
            ]
        )
    )

    # ==========================================================
    # Snapshots à évaluer
    # ==========================================================

    snapshots = range(1, 13)

    metrics = []

    # ==========================================================
    # Évaluation
    # ==========================================================

    for snap in snapshots:

        # -----------------------------------------
        # Tous les patients (événements + censures)
        # -----------------------------------------

        snapshot_all = (
            snapshot_period
            .filter(
                pl.col("n_measures") == snap
            )
            .select("patient_id")
            .unique()
        )

        n_patients = snapshot_all.height

        # Si ce snapshot n'existe pas
        if n_patients == 0:
            continue

        # -----------------------------------------
        # Patients ayant un événement
        # -----------------------------------------

        df = (
            results
            .filter(
                pl.col("n_measures") == snap
            )
        )

        n_events = df.height

        # -----------------------------------------
        # Patients censurés
        # -----------------------------------------

        n_censored = n_patients - n_events

        # -----------------------------------------
        # Aucun événement
        # -----------------------------------------

        if n_events == 0:
            continue

        # -----------------------------------------
        # Distribution des classes
        # -----------------------------------------

        class_counts = (
            df
            .group_by("interval_label")
            .len()
            .to_dict(as_series=False)
        )

        distribution = dict(
            zip(
                class_counts["interval_label"],
                class_counts["len"],
            )
        )

        # -----------------------------------------
        # Labels
        # -----------------------------------------

        y_true = df["interval_id"].to_numpy()
        y_pred = df["predicted_interval_id"].to_numpy()

        # -----------------------------------------
        # Nombre de bonnes prédictions
        # -----------------------------------------

        n_predicted = int((y_true == y_pred).sum())

        # -----------------------------------------
        # Balanced Accuracy
        # -----------------------------------------

        if np.unique(y_true).size > 1:

            balanced_acc = balanced_accuracy_score(
                y_true,
                y_pred,
            )

        else:

            balanced_acc = np.nan

        # -----------------------------------------
        # Métriques
        # -----------------------------------------

        metrics.append(
            {
                "n_measures": snap,
                "n_patients": n_patients,
                "n_censored": n_censored,
                "n_events": n_events,
                "n_predicted": n_predicted,
                "accuracy": accuracy_score(
                    y_true,
                    y_pred,
                ),
                "balanced_accuracy": balanced_acc,
                "macro_f1": f1_score(
                    y_true,
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
                "weighted_f1": f1_score(
                    y_true,
                    y_pred,
                    average="weighted",
                    zero_division=0,
                ),
            }
        )

    return (
        pl.DataFrame(metrics)
        .sort("n_measures")
    )

In [ ]:
# ==========================================================
# RÉSULTATS PAR SNAPSHOT
# ==========================================================

snapshot_results = evaluate_survival_stacking_by_snapshot(
    truth=truth,
    predictions=predictions,
    snapshot_period=internal_test_snapshot_period,
)

snapshot_results

### Analyse des prédictions

In [ ]:
# ==========================================================
# ÉVALUATION GLOBALE DES PERFORMANCES
# ==========================================================

def evaluate_survival_stacking_global(
    truth: pl.DataFrame,
    predictions: pl.DataFrame,
    snapshot_period: pl.DataFrame,
):
    """
    Évalue les performances globales du Survival Stacking
    sur l'ensemble des événements du jeu de test.

    Parameters
    ----------
    truth : pl.DataFrame
        Vérités terrain (patients ayant présenté un événement).

    predictions : pl.DataFrame
        Prédictions du modèle.

    snapshot_period : pl.DataFrame
        Jeu Snapshot-Period complet
        (événements + censures).

    Returns
    -------
    metrics : dict
    Les métriques sont calculées sur l'ensemble des prédictions
    effectuées pour les patients ayant présenté un événement.
    Un même patient peut donc contribuer avec plusieurs prédictions
    correspondant à différents snapshots.

    results : pl.DataFrame
    """

    # ==========================================================
    # Fusion vérité / prédictions
    # ==========================================================

    results = (

        truth

        .join(

            predictions,

            on=[
                "patient_id",
                "n_measures",
            ],

            how="inner",

        )

        .sort(
            [
                "n_measures",
                "patient_id",
            ]
        )

    )

    # ==========================================================
    # Population évaluée
    # ==========================================================

    n_patients = (

        snapshot_period

        .select("patient_id")

        .unique()

        .height

    )

    n_events = (

        truth

        .select("patient_id")

        .unique()

        .height

    )

    n_censored = n_patients - n_events

    # ==========================================================
    # Labels
    # ==========================================================

    y_true = results["interval_id"].to_numpy()

    y_pred = results["predicted_interval_id"].to_numpy()

    # ==========================================================
    # Métriques globales
    # ==========================================================

    metrics = {

        # ==========================================
        # Population du jeu de test
        # ==========================================

        "test_patients": n_patients,

        "test_event_patients": n_events,

        "test_censored_patients": n_censored,

        # ==========================================
        # Evaluation
        # ==========================================

        "evaluated_predictions": len(y_true),

        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

    }

    return metrics, results

In [ ]:
# ==========================================================
# PERFORMANCES GLOBALES
# ==========================================================

global_metrics, global_results = evaluate_survival_stacking_global(

    truth=truth,

    predictions=predictions,

    snapshot_period=internal_test_snapshot_period,

)

pl.DataFrame([global_metrics])

**Interprétation :**

  Les performances globales du modèle ont été évaluées sur le jeu de test interne, qui comprend 411 patients, dont 127 ont présenté un événement d'insuffisance rénale terminale (IRT) au cours du suivi et 284 ont été censurés. Les patients censurés ont été utilisés lors de l'entraînement du modèle de Survival Stacking, mais n'ont pas été inclus dans l'évaluation de la classification, l'intervalle réel de survenue de l'IRT étant inconnu. Au total, 506 prédictions longitudinales ont été évaluées, un même patient pouvant contribuer avec plusieurs prédictions correspondant aux différents snapshots disponibles au cours de son suivi.

Le modèle obtient une accuracy de 90,3 %, indiquant que plus de neuf prédictions longitudinales sur dix sont correctement classées dans l'intervalle de temps avant l'IRT correspondant. La balanced accuracy atteint 86,7 %, ce qui montre que les performances restent élevées après correction du déséquilibre entre les classes et que le modèle conserve une bonne capacité de discrimination pour chacune des catégories de prédiction.

Les scores F1 confirment ces résultats. Le Macro F1 de 88,6 % traduit une performance globalement homogène sur l'ensemble des classes, tandis que le Weighted F1 de 90,1 %, très proche de l'accuracy, indique que le modèle maintient également de bonnes performances lorsque la fréquence des différentes classes est prise en compte. L'écart limité entre ces différentes métriques suggère que le modèle ne repose pas uniquement sur les classes les plus représentées, mais présente une capacité de généralisation satisfaisante sur l'ensemble des intervalles de prédiction.


In [ ]:
# ==========================================================
# MATRICE DE CONFUSION
# ==========================================================

# Libellés des classes
CLASS_NAMES = [
    "<5 ans",
    "5–10 ans",
    "10–15 ans",
    ">15 ans",
]

# Construction du tableau de confusion
cm_df = (
    pl.DataFrame(
        metrics["confusion_matrix"],
        schema=CLASS_NAMES,
    )
    .with_columns(
        pl.Series("Classe réelle", CLASS_NAMES)
    )
    .select(["Classe réelle"] + CLASS_NAMES)
)

# Affichage
cm_df

In [ ]:
fig = px.imshow(

    cm_df.select(CLASS_NAMES).to_pandas(),

    x=CLASS_NAMES,

    y=CLASS_NAMES,

    text_auto=True,

    color_continuous_scale="Blues",

    labels=dict(

        x="Classe prédite",

        y="Classe réelle",

        color="Effectif",

    ),

)

fig.update_layout(

    title="Matrice de confusion",

)

fig.show()

In [ ]:
# ==========================================================
# MATRICE DE CONFUSION NORMALISÉE
# ==========================================================

cm_percent = (
    cm_df
    .select(CLASS_NAMES)
    .to_pandas()
    .astype(float)
)

cm_percent = cm_percent.div(
    cm_percent.sum(axis=1),
    axis=0,
)

fig = px.imshow(

    cm_percent,

    x=CLASS_NAMES,

    y=CLASS_NAMES,

    text_auto=".1%",

    color_continuous_scale="Blues",

    labels=dict(

        x="Classe prédite",

        y="Classe réelle",

        color="Proportion",

    ),

)

fig.update_layout(

    title="Matrice de confusion normalisée",

)

fig.show()

La matrice de confusion montre que le modèle distingue correctement les quatre intervalles de temps avant l'IRT. 
La classe <5 ans est parfaitement identifiée (100 % de bonnes prédictions), tandis que les classes 5–10 ans, 10–15 ans et >15 ans présentent respectivement des taux de bonnes prédictions de 95,8 %, 77,4 % et 74,1 %. Les erreurs observées concernent essentiellement des classes temporellement voisines, sans confusion entre les intervalles les plus éloignés. Cette observation suggère que, lorsqu'une erreur est commise, le modèle estime généralement un intervalle proche de l'intervalle réel.


In [ ]:
# ==========================================================
# MATRICE DE CONFUSION PAR SNAPSHOT
# ==========================================================

CLASS_NAMES = [
    "<5 ans",
    "5–10 ans",
    "10–15 ans",
    ">15 ans",
]

# ==========================================================
# Résultats (vérité + prédictions)
# ==========================================================

metrics, results = evaluate_survival_stacking_predictions(
    truth=truth,
    predictions=predictions,
)

# ==========================================================
# Fonction
# ==========================================================

def confusion_matrix_snapshot(snapshot):

    df = (
        results
        .filter(
            pl.col("n_measures") == snapshot
        )
    )

    cm = confusion_matrix(
        df["interval_id"],
        df["predicted_interval_id"],
        labels=[0, 1, 2, 3],
    )

    cm_df = (
        pl.DataFrame(
            cm,
            schema=CLASS_NAMES,
        )
        .with_columns(
            pl.Series(
                "Classe réelle",
                CLASS_NAMES,
            )
        )
        .select(
            ["Classe réelle"] + CLASS_NAMES
        )
    )

    display(cm_df)

# ==========================================================
# Onglets
# ==========================================================

tabs = widgets.Tab()

outputs = []

for snap in range(1, 13):

    out = widgets.Output()

    with out:
        confusion_matrix_snapshot(snap)

    outputs.append(out)

tabs.children = outputs

for i in range(12):
    tabs.set_title(i, f"Snapshot {i+1}")

display(tabs)

In [ ]:
# ==========================================================
# RAPPORT DE CLASSIFICATION
# ==========================================================

# Construction du rapport de classification
report_df = (
    pd.DataFrame(metrics["classification_report"])
    .T
    .reset_index()
    .rename(columns={"index": "Classe"})
)

# Remplacement des indices numériques par les noms des classes
report_df.loc[0:3, "Classe"] = CLASS_NAMES

# Conservation des quatre classes uniquement
report_df = (
    pl.from_pandas(report_df)
    .filter(pl.col("Classe").is_in(CLASS_NAMES))
)

# Affichage
report_df

**Interprétation :**

Le rapport de classification confirme les bonnes performances du modèle sur l'ensemble des classes. Les classes <5 ans et 5–10 ans présentent les meilleurs résultats, avec des F1-scores respectifs de 0,948 et 0,921. Les performances sont légèrement plus faibles pour les classes 10–15 ans (F1-score = 0,811) et >15 ans (F1-score = 0,863), principalement en raison d'un rappel plus faible. Ces résultats sont cohérents avec la matrice de confusion, qui montre que les erreurs concernent essentiellement des classes temporellement adjacentes.

In [ ]:
# ==========================================================
# ÉVOLUTION DES PERFORMANCES PAR SNAPSHOT
# ==========================================================

# Limiter l'analyse aux 12 premiers snapshots
plot_df = snapshot_results.filter(
    pl.col("n_measures") <= 12
)

fig = go.Figure()

# ----------------------------------------------------------
# Performances (axe de gauche)
# ----------------------------------------------------------

performance_metrics = [
    ("accuracy", "Accuracy"),
    ("balanced_accuracy", "Balanced Accuracy"),
    ("macro_f1", "Macro F1-score"),
]

for column, label in performance_metrics:

    fig.add_trace(
        go.Scatter(
            x=plot_df["n_measures"],
            y=plot_df[column],
            mode="lines+markers",
            name=label,
            yaxis="y1",
        )
    )

# ----------------------------------------------------------
# Effectifs (axe de droite)
# ----------------------------------------------------------

fig.add_trace(
    go.Bar(
        x=plot_df["n_measures"],
        y=plot_df["n_patients"],
        name="Patients",
        yaxis="y2",
        opacity=0.25,
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["n_measures"],
        y=plot_df["n_events"],
        mode="lines+markers",
        name="Événements",
        yaxis="y2",
        line=dict(dash="dash"),
    )
)

# ----------------------------------------------------------
# Mise en forme
# ----------------------------------------------------------

fig.update_layout(

    title="Évolution des performances selon le nombre de mesures",

    template="simple_white",

    xaxis=dict(
        title="Nombre de mesures disponibles",
        dtick=1,
    ),

    yaxis=dict(
        title="Performance",
        range=[0.70, 1.02],
    ),

    yaxis2=dict(
        title="Nombre de patients",
        overlaying="y",
        side="right",
    ),

    legend=dict(
        orientation="h",
        y=1.15,
    ),

    bargap=0.35,

)

fig.show()

## INTERPRÉTABILITÉ DU MODÈLE AVEC SHAP

In [ ]:
# ==========================================================
# INTERPRÉTABILITÉ DU MODÈLE AVEC SHAP
# ==========================================================


# ==========================================================
# SHAP EXPLAINER
# ==========================================================

explainer = shap.TreeExplainer(model)

In [ ]:
# ==========================================================
# FONCTIONS UTILITAIRES POUR L'ANALYSE SHAP
# ==========================================================

def get_snapshot_data(snapshot: int):
    """
    Retourne toutes les informations nécessaires
    pour analyser un snapshot.
    """

    # ======================================================
    # Données utilisées pour SHAP
    # ======================================================

    mask = X_test["n_measures"] == snapshot

    # IMPORTANT :
    # On conserve exactement les variables utilisées
    # lors de l'entraînement du modèle.
    X_snapshot = X_test.loc[mask].copy()

    y_snapshot = y_test.loc[mask].copy()

    # ======================================================
    # Données Survival Stacking
    # ======================================================

    snapshot_period = (
        internal_test_snapshot_period
        .filter(
            pl.col("n_measures") == snapshot
        )
    )

    # ======================================================
    # Performances du snapshot
    # ======================================================

    row = (
        snapshot_results
        .filter(
            pl.col("n_measures") == snapshot
        )
    )

    metrics = (
        row.to_dicts()[0]
        if row.height > 0
        else None
    )

    return (
        X_snapshot,
        y_snapshot,
        snapshot_period,
        metrics,
    )


# ==========================================================
# STATISTIQUES DESCRIPTIVES
# ==========================================================

def snapshot_statistics(snapshot_period: pl.DataFrame):
    """
    Calcule les statistiques descriptives
    d'un snapshot.
    """

    n_patients = (
        snapshot_period
        .select("patient_id")
        .unique()
        .height
    )

    n_events = (
        snapshot_period
        .filter(
            pl.col("event_irt") == 1
        )
        .select("patient_id")
        .unique()
        .height
    )

    n_censored = n_patients - n_events

    return {

        "n_patients": n_patients,

        "n_events": n_events,

        "n_censored": n_censored,

    }


# ==========================================================
# IMPORTANCE GLOBALE DES VARIABLES
# ==========================================================

def compute_shap_importance(
    shap_values,
    feature_names,
):
    """
    Calcule l'importance moyenne absolue
    des valeurs SHAP.
    """

    importance = np.abs(
        shap_values.values
    ).mean(axis=0)

    importance_df = (
        pd.DataFrame(
            {
                "Feature": list(feature_names),
                "Mean |SHAP|": importance,
            }
        )
        .sort_values(
            "Mean |SHAP|",
            ascending=False,
            ignore_index=True,
        )
    )

    return importance_df


# ==========================================================
# CALCUL DES VALEURS SHAP
# ==========================================================

def compute_snapshot_shap(snapshot: int):
    """
    Calcule les valeurs SHAP d'un snapshot.
    """

    (
        X_snapshot,
        _,
        _,
        _,
    ) = get_snapshot_data(snapshot)

    if X_snapshot.empty:
        raise ValueError(
            f"Aucune observation disponible pour le snapshot {snapshot}."
        )

    shap_values = explainer(X_snapshot)

    importance_df = compute_shap_importance(
        shap_values,
        X_snapshot.columns,
    )

    return (
        X_snapshot,
        shap_values,
        importance_df,
    )

In [ ]:
# ==========================================================
# WIDGET STATISTIQUES ET PERFORMANCES
# ==========================================================

snapshot_values = sorted(
    s

    for s in range(1, 13)

    if s in X_test["n_measures"].unique()
)

snapshot_selector_stats = widgets.Dropdown(

    options=snapshot_values,

    value=snapshot_values[0],

    description="Snapshot",

    style={
        "description_width": "initial",
    },

    layout=widgets.Layout(
        width="220px",
    ),

)

output_stats = widgets.Output()


def update_statistics(snapshot):

    with output_stats:

        clear_output(wait=True)

        (
            _,
            _,
            snapshot_period,
            metrics,
        ) = get_snapshot_data(snapshot)

        stats = snapshot_statistics(
            snapshot_period
        )

        print("=" * 70)
        print(f"Snapshot {snapshot}")
        print("=" * 70)
        print()

        print("Population")
        print(f"  Nombre de patients   : {stats['n_patients']}")
        print(f"  Nombre d'évènements  : {stats['n_events']}")
        print(f"  Nombre censurés      : {stats['n_censored']}")
        print()

        if metrics is not None:

            print("Performances")

            if "accuracy" in metrics:
                print(f"  Accuracy            : {metrics['accuracy']:.3f}")

            if "balanced_accuracy" in metrics:
                print(f"  Balanced Accuracy   : {metrics['balanced_accuracy']:.3f}")

            if "macro_f1" in metrics:
                print(f"  Macro F1            : {metrics['macro_f1']:.3f}")

            if "weighted_f1" in metrics:
                print(f"  Weighted F1         : {metrics['weighted_f1']:.3f}")

In [ ]:
# ==========================================================
# LANCEMENT
# ==========================================================

dashboard_stats = widgets.interactive_output(

    update_statistics,

    {

        "snapshot": snapshot_selector_stats,

    },

)

display(

    widgets.VBox(

        [

            snapshot_selector_stats,

            output_stats,

            dashboard_stats,

        ]

    )

)

In [ ]:
# ==========================================================
# WIDGET SHAP BEESWARM
# ==========================================================

snapshot_values = [

    s

    for s in range(1, 13)

    if s in X_test["n_measures"].unique()

]

snapshot_selector_beeswarm = widgets.Dropdown(

    options=snapshot_values,

    value=snapshot_values[0],

    description="Snapshot",

    style={
        "description_width": "initial",
    },

    layout=widgets.Layout(
        width="220px",
    ),

)

output_beeswarm = widgets.Output()


def update_beeswarm(snapshot):

    with output_beeswarm:

        clear_output(wait=True)

        # ==================================================
        # Calcul des valeurs SHAP
        # ==================================================

        (
            X_snapshot,
            shap_values,
            _,
        ) = compute_snapshot_shap(snapshot)

        # ==================================================
        # Variables affichées
        # ==================================================

        features_to_plot = [

            col

            for col in X_snapshot.columns

            if col != "interval_id"

        ]

        feature_idx = [

            X_snapshot.columns.get_loc(col)

            for col in features_to_plot

        ]

        shap_values_plot = shap_values[:, feature_idx]

        # ==================================================
        # Affichage
        # ==================================================

        print("=" * 70)
        print(f"SHAP Beeswarm - Snapshot {snapshot}")
        print("=" * 70)
        print()

        shap.plots.beeswarm(

            shap_values_plot,

            max_display=20,

            show=False,

        )

        plt.tight_layout()

        plt.show()

In [ ]:
# sauvegarde un beeswarm d'un sanp particulier

snapshot = 3

X_snapshot, shap_values, _ = compute_snapshot_shap(snapshot)

features_to_plot = [
    col
    for col in X_snapshot.columns
    if col != "interval_id"
]

feature_idx = [
    X_snapshot.columns.get_loc(col)
    for col in features_to_plot
]

shap_values_plot = shap_values[:, feature_idx]

shap.plots.beeswarm(
    shap_values_plot,
    max_display=20,
    show=False,
)

plt.title(
    f"SHAP Beeswarm - Snapshot {snapshot}",
    fontsize=14,
    fontweight="bold",
)

plt.tight_layout()

plt.savefig(
    "shap_beeswarm_snapshot3.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ==========================================================
# LANCEMENT DU WIDGET
# ==========================================================

dashboard_beeswarm = widgets.interactive_output(

    update_beeswarm,

    {

        "snapshot": snapshot_selector_beeswarm,

    },

)

display(

    widgets.VBox(

        [

            snapshot_selector_beeswarm,

            output_beeswarm,

            dashboard_beeswarm,

        ]

    )

)

In [ ]:
# ==========================================================
# WIDGET SHAP FEATURE IMPORTANCE
# ==========================================================

snapshot_values = [

    s

    for s in range(1, 13)

    if s in X_test["n_measures"].unique()

]

snapshot_selector_bar = widgets.Dropdown(

    options=snapshot_values,

    value=snapshot_values[0],

    description="Snapshot",

    style={"description_width": "initial"},

    layout=widgets.Layout(width="220px"),

)

output_bar = widgets.Output()


def update_bar(snapshot):

    with output_bar:

        clear_output(wait=True)

        (
            X_snapshot,
            shap_values,
            _,
        ) = compute_snapshot_shap(snapshot)

        features_to_plot = [

            col

            for col in X_snapshot.columns

            if col != "interval_id"

        ]

        feature_idx = [

            X_snapshot.columns.get_loc(col)

            for col in features_to_plot

        ]

        shap_values_plot = shap_values[:, feature_idx]

        importance = compute_shap_importance(
            shap_values,
            X_snapshot.columns
        )

        display(importance.head(20))

        shap.plots.bar(

            shap_values_plot,

            max_display=15,

            show=False,

        )

        ax = plt.gca()

        # Barres en bleu
        for bar in ax.patches:

            bar.set_facecolor("steelblue")

        # Annotations numériques en noir
        for text in ax.texts:

            text.set_color("black")

        ax.set_title(

            f"SHAP Feature Importance - Snapshot {snapshot}",

            fontsize=14,

            fontweight="bold",

        )

        plt.tight_layout()

        plt.show()

In [ ]:
# ==========================================================
# LANCEMENT DU WIDGET
# ==========================================================

dashboard_bar = widgets.interactive_output(

    update_bar,

    {

        "snapshot": snapshot_selector_bar,

    },

)

display(

    widgets.VBox(

        [

            snapshot_selector_bar,

            output_bar,

            dashboard_bar,

        ]

    )

)

In [ ]:
# ==========================================================
# WIDGET : PATIENTS MAL CLASSÉS
# ==========================================================

def display_misclassified(snapshot):

    # ======================================================
    # Résultats (vérité + prédictions)
    # ======================================================

    _, results = evaluate_survival_stacking_predictions(
        truth=truth,
        predictions=predictions,
    )

    # ======================================================
    # Ajout des informations cliniques
    # ======================================================

    patient_info = (
        internal_test_snapshot_period
        .select(
            [
                "patient_id",
                "n_measures",
                "age_creat",
                "sex",
                "genetic",
            ]
        )
        .unique(
            subset=[
                "patient_id",
                "n_measures",
            ]
        )
    )

    results = (
        results
        .join(
            patient_info,
            on=[
                "patient_id",
                "n_measures",
            ],
            how="left",
        )
    )

    # ======================================================
    # Filtre sur le snapshot
    # ======================================================

    errors = (
        results
        .filter(
            (pl.col("n_measures") == snapshot)
            &
            (pl.col("interval_id") != pl.col("predicted_interval_id"))
        )
        .with_columns(

            (
                pl.col("predicted_interval_id")
                - pl.col("interval_id")
            ).alias("Décalage"),

            pl.when(
                pl.col("predicted_interval_id")
                > pl.col("interval_id")
            )
            .then(pl.lit("Sur-estimation"))
            .otherwise(pl.lit("Sous-estimation"))
            .alias("Type d'erreur")

        )
        .select(
            [
                "patient_id",
                "n_measures",
                "age_creat",
                "sex",
                "genetic",
                "interval_label",
                "predicted_interval_label",
                "predicted_probability",
                "Décalage",
                "Type d'erreur",
            ]
        )
        .rename(
            {
                "patient_id": "Patient",
                "n_measures": "Snapshot",
                "age_creat": "Âge",
                "sex": "Sexe",
                "genetic": "Génétique",
                "interval_label": "Classe réelle",
                "predicted_interval_label": "Classe prédite",
                "predicted_probability": "Probabilité",
            }
        )
        .sort(
            [
                "Classe réelle",
                "Patient",
            ]
        )
    )

    print(f"Nombre de patients mal classés : {errors.height}")

    if errors.height == 0:
        print("Aucun patient mal classé.")
    else:
        display(errors)


# ==========================================================
# WIDGET
# ==========================================================

widgets.interact(
    display_misclassified,
    snapshot=widgets.Dropdown(
        options=sorted(
            truth["n_measures"].unique().to_list()
        ),
        value=1,
        description="Snapshot",
    ),
);

## VALIDATION EXTERNE

In [ ]:
# ================================================================
# PRÉPARATION DES MATRICES D'ENTRAÎNEMENT ET DE TEST EXTERNE
# ================================================================

(
    X_train,
    y_train,
    X_external,
    y_external,
    encoders,
) = prepare_snapshot_period_data(
    train_snapshot_period,
    external_test_snapshot_period,
    FEATURES,
)


In [ ]:
# ==========================================================
# PRÉDICTION SUR LE JEU DE TEST EXTERNE
# ==========================================================

hazards_external = predict_conditional_hazard(
    model=model,
    X=X_external,
    data_snapshot_period=external_test_snapshot_period,
)

survival_external = compute_survival_probabilities(
    hazards_external
)

predictions_external = predict_survival_interval(
    survival_external
)

truth_external = (
    external_test_snapshot_period
    .filter(
        pl.col("event_interval") == 1
    )
    .select(
        [
            "patient_id",
            "n_measures",
            "interval_id",
            "interval_label",
        ]
    )
    .unique(
        subset=[
            "patient_id",
            "n_measures",
        ]
    )
    .sort(
        [
            "patient_id",
            "n_measures",
        ]
    )
)

# ==========================================================
# ÉVALUATION SUR LE JEU DE TEST EXTERNE
# ==========================================================

metrics_external, results_external = (
    evaluate_survival_stacking_predictions(
        truth=truth_external,
        predictions=predictions_external,
    )
)

In [ ]:
# ==========================================================
# PRÉSENTATION DES RÉSULTATS
# ==========================================================

print("=" * 70)
print("SURVIVAL STACKING - RÉSULTATS (VALIDATION EXTERNE)")
print("=" * 70)

print(f"Accuracy              : {metrics_external['accuracy']:.4f}")
print(f"Balanced Accuracy     : {metrics_external['balanced_accuracy']:.4f}")
print(f"Macro F1-score        : {metrics_external['macro_f1']:.4f}")
print(f"Weighted F1-score     : {metrics_external['weighted_f1']:.4f}")

print("\n")

# ==========================================================
# Matrice de confusion
# ==========================================================

print("=" * 70)
print("MATRICE DE CONFUSION")
print("=" * 70)

print(metrics_external["confusion_matrix"])

print("\n")

# ==========================================================
# Rapport de classification
# ==========================================================

print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

report_external = (
    pd.DataFrame(metrics_external["classification_report"])
    .T
    .round(3)
)

print(report_external)

print("\n")

# ==========================================================
# Résultats détaillés
# ==========================================================

print("=" * 70)
print("APERÇU DES PRÉDICTIONS")
print("=" * 70)

print(results_external)

print("\n")

# ==========================================================
# Statistiques
# ==========================================================

print("=" * 70)
print("STATISTIQUES")
print("=" * 70)

print(f"Nombre de prédictions : {results_external.height}")

print(
    f"Nombre de patients    : "
    f"{results_external.select('patient_id').n_unique()}"
)

In [ ]:
# ==========================================================
# PRÉDICTIONS CORRECTES
# ==========================================================

results_external.with_columns(
    (
        pl.col("predicted_interval_id")
        == pl.col("interval_id")
    ).alias("correct_prediction")
)


In [ ]:
# ==========================================================
# PATIENTS CENSURÉS - VALIDATION EXTERNE
# ==========================================================

censor_metrics_external, censor_results_external = (
    evaluate_censoring_predictions(
        truth=external_test_snapshot_period,
        predictions=predictions_external,
    )
)

print("=" * 70)
print("ANALYSE DES PATIENTS CENSURÉS (VALIDATION EXTERNE)")
print("=" * 70)

print(f"Nombre de patients censurés      : {censor_metrics_external['n_censored']}")
print(f"Violations de la censure         : {censor_metrics_external['n_violations']}")
print(f"Taux de violation                : {100*censor_metrics_external['censoring_violation_rate']:.2f}%")
print(f"Cohérence avec la censure        : {100*censor_metrics_external['censoring_consistency']:.2f}%")

print("\n")

In [ ]:
# ==========================================================
# Tableau récapitulatif
# ==========================================================

summary_external = (

    censor_results_external

    .select(

        [

            "patient_id",

            "n_measures",

            "time_to_event",

            "predicted_interval_label",

            "predicted_probability",

            "violation",

        ]

    )

    .sort(

        [

            "violation",

            "time_to_event",

        ],

        descending=[True, False],

    )

)

print("=" * 70)
print("PATIENTS CENSURÉS")
print("=" * 70)

print(summary_external)

print("\n")

# ==========================================================
# Violations uniquement
# ==========================================================

violations_external = summary_external.filter(
    pl.col("violation")
)

print("=" * 70)
print("PATIENTS PRÉSENTANT UNE VIOLATION")
print("=" * 70)

if violations_external.height == 0:

    print("Aucune violation.")

else:

    print(violations_external)

# ==========================================================
# RÉSULTATS PAR SNAPSHOT
# VALIDATION EXTERNE
# ==========================================================

snapshot_results_external = evaluate_survival_stacking_by_snapshot(
    truth=truth_external,
    predictions=predictions_external,
    snapshot_period=external_test_snapshot_period,
)

snapshot_results_external

In [ ]:
# ==========================================================
# TABLEAU DES MÉTRIQUES
# ==========================================================

metrics_df = pd.DataFrame({

    "Metric": [

        "Accuracy",
        "Balanced Accuracy",
        "Macro F1-score",
        "Weighted F1-score",

    ],

    "Internal Validation": [

        metrics["accuracy"],
        metrics["balanced_accuracy"],
        metrics["macro_f1"],
        metrics["weighted_f1"],

    ],

    "External Validation": [

        metrics_external["accuracy"],
        metrics_external["balanced_accuracy"],
        metrics_external["macro_f1"],
        metrics_external["weighted_f1"],

    ],

}).round(4)

In [ ]:
# ==========================================================
# SAUVEGARDE HTML
# SURVIVAL STACKING
# VALIDATION INTERNE & EXTERNE
# ==========================================================

import os

# ==========================================================
# DOSSIER DE SORTIE
# ==========================================================

output_dir = (
    r".\tables\survival_models"
)

os.makedirs(output_dir, exist_ok=True)

# ==========================================================
# CHEMIN HTML
# ==========================================================

html_path = os.path.join(
    output_dir,
    "survival_stacking_internal_external.html"
)

# ==========================================================
# GÉNÉRATION HTML
# ==========================================================

html_content = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="utf-8">

<title>Survival Stacking Results</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 40px;
    background-color: white;
}}

h1 {{
    color: #222222;
}}

h2 {{
    margin-top: 45px;
    color: #333333;
}}

h3 {{
    margin-top: 25px;
    color: #555555;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 15px;
    margin-bottom: 35px;
}}

th,
td {{
    border: 1px solid #cccccc;
    padding: 8px;
    text-align: center;
}}

th {{
    background-color: #f2f2f2;
}}

tr:nth-child(even) {{
    background-color: #fafafa;
}}

</style>

</head>

<body>

<h1>Survival Stacking Results</h1>

<h2>Overall Metrics</h2>

{metrics_df.to_html(index=False)}

<h2>Results by Snapshot</h2>

<h3>Internal Validation</h3>

{snapshot_results.to_pandas().to_html(index=False)}

<h3>External Validation</h3>

{snapshot_results_external.to_pandas().to_html(index=False)}

<h2>Censoring Analysis</h2>

{violations_df.to_html(index=False)}

</body>

</html>
"""

# ==========================================================
# SAUVEGARDE
# ==========================================================

with open(
    html_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(html_content)

print(f"HTML sauvegardé :\n{html_path}")